# Phase 2 — Initial Raw Data Audit

Exploratory profiling of the immutable SEC raw SQLite layer **before** any cleaning, scoring, or alpha modeling.

**This notebook does not:** filter P/S, normalize titles, remove rows, or build clean tables.

Reusable logic lives in `src/analytics/raw_data_audit.py`. Exports go to `data/exports/raw_data_audit/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics import raw_data_audit as audit
from src.utils.db import get_connection, get_database_path

audit.configure_logging()
EXPORT_DIR = audit.ensure_export_dirs()
DB_PATH = get_database_path()
conn = get_connection(DB_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Database:     {DB_PATH}")
print(f"Exports:      {EXPORT_DIR}")

Project root: C:\Users\artkh\Downloads\INSIDER ALPHA LAB\insider-research
Database:     C:\Users\artkh\Downloads\INSIDER ALPHA LAB\insider-research\data\processed\insider_research.db
Exports:      C:\Users\artkh\Downloads\INSIDER ALPHA LAB\insider-research\data\exports\raw_data_audit


## Section 1 — Database overview

Row counts per table, approximate database size, and date/year coverage where available.

In [2]:
overview = audit.get_database_overview(conn, DB_PATH)
audit.export_csv(overview, "table_overview.csv", EXPORT_DIR)
display(overview)
print(f"DB size: {overview['db_size_mb'].iloc[0]:.2f} MB")

12:47:35 | INFO | exported table_overview.csv (8 rows)


,table_name,row_count,min_date,max_date,db_size_mb,min_source_year,max_source_year,distinct_years
0,raw_deriv_holding,1741164,None,None,15154.67,2006,2026,21
1,raw_deriv_trans,2708249,None,None,15154.67,2006,2026,21
2,raw_footnotes,9269692,None,None,15154.67,2006,2026,21
3,raw_nonderiv_holding,2367536,None,None,15154.67,2006,2026,21
4,raw_nonderiv_trans,7215506,None,None,15154.67,2006,2026,21
5,raw_owner_signature,4676762,None,None,15154.67,2006,2026,21
6,raw_reporting_owner,4772838,None,None,15154.67,2006,2026,21
7,raw_submission,4402307,None,None,15154.67,2006,2026,21


DB size: 15154.67 MB


## Section 2 — Filing types (`raw_submission`)

Document type mix, filings per calendar year, and filings per SEC source quarter.

In [3]:
filing = audit.analyze_filing_types(conn)
audit.export_csv(filing["filing_type_distribution"], "filing_type_distribution.csv", EXPORT_DIR)
audit.plot_filing_charts(filing, EXPORT_DIR)

display(filing["document_type_distribution"].head(15))
display(filing["filings_by_year"].tail(10))

12:47:45 | INFO | exported filing_type_distribution.csv (87 rows)
12:47:45 | INFO | saved chart filing_type_distribution.png


,document_type,row_count,pct
0,4,3884991,88.2490
1,3,325974,7.4046
2,4/A,98691,2.2418
3,5,74596,1.6945
4,3/A,15607,0.3545
5,5/A,2448,0.0556


,filing_year,filing_count


## Section 3 — Transaction code analysis

Non-derivative + derivative transaction codes. Focus: **P, S, A, M, F, D, G, C, X**.

In [4]:
trans = audit.analyze_transaction_codes(conn)
audit.export_csv(trans["transaction_code_distribution"], "transaction_code_distribution.csv", EXPORT_DIR)
audit.plot_transaction_code_charts(trans, EXPORT_DIR)

display(trans["transaction_code_distribution"].head(20))
display(trans["focus_codes"])

12:48:34 | INFO | exported transaction_code_distribution.csv (18 rows)
12:48:34 | INFO | saved chart transaction_codes_over_time.png
12:48:34 | INFO | saved chart transaction_code_distribution.png


,transaction_code,row_count,pct,is_focus_code
0,S,2627131,26.4732,True
1,A,2478665,24.9771,True
2,M,1958139,19.7319,True
3,P,893817,9.0069,True
4,F,822342,8.2866,True
5,J,350831,3.5353,False
6,D,312811,3.1521,True
7,G,200883,2.0243,True
8,C,167578,1.6887,True
9,X,49607,0.4999,True


,transaction_code,row_count,pct
0,S,2627131,26.4732
1,A,2478665,24.9771
2,M,1958139,19.7319
3,P,893817,9.0069
4,F,822342,8.2866
6,D,312811,3.1521
7,G,200883,2.0243
8,C,167578,1.6887
9,X,49607,0.4999


## Section 4 — Security type analysis

`SECURITY_TITLE` frequency, common-stock variants, and derivative/option prevalence.

In [5]:
security = audit.analyze_security_titles(conn)
audit.export_csv(security["security_type_distribution"], "security_type_distribution.csv", EXPORT_DIR)

display(security["security_type_distribution"].head(25))
display(security["normalized_preview"])

12:49:07 | INFO | exported security_type_distribution.csv (100 rows)


,security_title,row_count,pct
0,Common Stock,4714847,47.5107
1,Class A Common Stock,667781,6.7291
2,Restricted Stock Units,372434,3.7530
3,Stock Option (Right to Buy),177176,1.7854
4,Common Shares,154084,1.5527
5,Stock Option (right to buy),127897,1.2888
6,Class B Common Stock,114009,1.1488
7,Non-Qualified Stock Option (right to buy),103882,1.0468
8,Restricted Stock Unit,98389,0.9914
9,Employee Stock Option (right to buy),97411,0.9816


,security_title,row_count,pct,normalized_preview
0,Common Stock,4714847,47.5107,COMMON STOCK
1,Class A Common Stock,667781,6.7291,CLASS A COMMON STOCK
2,Restricted Stock Units,372434,3.7530,RESTRICTED STOCK UNITS
3,Stock Option (Right to Buy),177176,1.7854,STOCK OPTION (RIGHT TO BUY)
4,Common Shares,154084,1.5527,COMMON SHARES
5,Stock Option (right to buy),127897,1.2888,STOCK OPTION (RIGHT TO BUY)
6,Class B Common Stock,114009,1.1488,CLASS B COMMON STOCK
7,Non-Qualified Stock Option (right to buy),103882,1.0468,NON-QUALIFIED STOCK OPTION (RIGHT TO BUY)
8,Restricted Stock Unit,98389,0.9914,RESTRICTED STOCK UNIT
9,Employee Stock Option (right to buy),97411,0.9816,EMPLOYEE STOCK OPTION (RIGHT TO BUY)


## Transaction Code Taxonomy (Phase 2.2b)

SEC `TRANS_CODE` values are **not** interchangeable. Most rows are compensation, derivative exercises, or tax events — not discretionary open-market trades.

This section applies the config-driven taxonomy in `configs/normalization/transaction_code_map.csv` to `raw_nonderiv_trans` only (raw codes preserved).

**MVP research focus:** codes **P** (purchase) and **S** (sale) — high/medium signal quality open-market events. Phase 3 will filter using `include_clean_mvp`; we do **not** filter the database here.

In [6]:
from src.utils.sec_taxonomy import load_transaction_code_map

taxonomy_map = load_transaction_code_map()
display(taxonomy_map)

tax = audit.analyze_transaction_code_taxonomy(conn, export_dir=EXPORT_DIR)
summary = tax["summary"]

print("Taxonomy coverage summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

display(tax["coverage"].head(25))
print("\nUnmapped codes (if any):")
display(tax["unmapped"])

mvp_rows = tax["coverage"].loc[tax["coverage"]["include_clean_mvp"] == True, ["transaction_code", "row_count", "pct_total", "category", "signal_quality"]]
display(mvp_rows)

,transaction_code,meaning,category,signal_direction,signal_quality,include_clean_mvp,requires_special_handling,economic_interpretation,notes
0,P,Open market or private purchase,OPEN_MARKET,BULLISH,HIGH,True,False,Discretionary capital allocation by insider; p...,SEC Table 1 code P
1,S,Open market or private sale,OPEN_MARKET,BEARISH_OR_NEUTRAL,MEDIUM,True,False,Discretionary sale; informative but noisier th...,SEC Table 1 code S
2,A,Grant award or other acquisition from issuer,COMPENSATION,NONE,LOW,False,True,Compensation or plan grant; not a market-timin...,Often RSU/option grants; exclude from MVP
3,M,Exercise or conversion of derivative security,DERIVATIVE_EVENT,NONE,LOW,False,True,Mechanical derivative exercise; often paired w...,SEC Table 1 code M
4,F,Payment of exercise price or tax liability wit...,TAX_EVENT,NONE,VERY_LOW,False,True,Tax withholding or cashless exercise mechanics...,Often follows M or A
5,D,Sale or transfer of securities back to issuer,CORPORATE_ACTION,NONE,VERY_LOW,False,True,Issuer repurchase or return of shares; adminis...,SEC Table 1 code D
6,G,Gift of securities by or to insider,TRANSFER,NONE,VERY_LOW,False,True,Non-market transfer; no price discovery,SEC Table 1 code G
7,C,Conversion of derivative security,DERIVATIVE_EVENT,NONE,LOW,False,True,Structural conversion; not open-market decision,SEC Table 1 code C
8,X,Exercise of in-the-money or at-the-money deriv...,DERIVATIVE_EVENT,NONE,LOW,False,True,Option exercise event; may precede open-market...,SEC Table 1 code X
9,J,Other acquisition or disposition,ADMINISTRATIVE,MIXED,LOW,False,True,Catch-all; requires footnote review,SEC Table 1 code J


12:49:12 | INFO | exported transaction_code_taxonomy_coverage.csv (18 rows)
12:49:12 | INFO | saved chart transaction_code_categories.png
12:49:12 | INFO | saved chart signal_quality_distribution.png
12:49:12 | INFO | taxonomy coverage: 100.00% (7,215,500 / 7,215,500 rows), 0 unmapped codes


Taxonomy coverage summary:
  total_rows: 7215500
  distinct_codes: 18
  mapped_codes: 18
  unmapped_codes: 0
  rows_mapped: 7215500
  rows_unmapped: 0
  coverage_pct: 100.0


,transaction_code,row_count,pct_total,mapped,category,signal_quality,include_clean_mvp,meaning,signal_direction
0,S,2613025,36.2141,True,OPEN_MARKET,MEDIUM,True,Open market or private sale,BEARISH_OR_NEUTRAL
1,A,1235992,17.1297,True,COMPENSATION,LOW,False,Grant award or other acquisition from issuer,NONE
2,M,954686,13.2310,True,DERIVATIVE_EVENT,LOW,False,Exercise or conversion of derivative security,NONE
3,P,856821,11.8747,True,OPEN_MARKET,HIGH,True,Open market or private purchase,BULLISH
4,F,815743,11.3054,True,TAX_EVENT,VERY_LOW,False,Payment of exercise price or tax liability wit...,NONE
5,J,258394,3.5811,True,ADMINISTRATIVE,LOW,False,Other acquisition or disposition,MIXED
6,G,187978,2.6052,True,TRANSFER,VERY_LOW,False,Gift of securities by or to insider,NONE
7,D,144619,2.0043,True,CORPORATE_ACTION,VERY_LOW,False,Sale or transfer of securities back to issuer,NONE
8,C,78985,1.0947,True,DERIVATIVE_EVENT,LOW,False,Conversion of derivative security,NONE
9,L,20635,0.2860,True,ADMINISTRATIVE,VERY_LOW,False,Small acquisition under Rule 16a-6,NONE



Unmapped codes (if any):


,transaction_code,row_count,pct_total,mapped,category,signal_quality,include_clean_mvp,meaning,signal_direction


,transaction_code,row_count,pct_total,category,signal_quality
0,S,2613025,36.2141,OPEN_MARKET,MEDIUM
3,P,856821,11.8747,OPEN_MARKET,HIGH


## Security Type Taxonomy (Phase 2.3)

`SECURITY_TITLE` is noisy (case, share class, compensation labels). This section classifies **distinct** titles in `raw_nonderiv_trans` using `configs/normalization/security_title_map.csv`.

**MVP universe:** `COMMON_EQUITY` with `include_clean_mvp=true` (true open-market common stock). Options, RSUs, phantom/deferred, preferred, and partnership units are excluded from MVP but preserved in raw data.

In [ ]:
sec_tax = audit.analyze_security_type_taxonomy(conn, export_dir=EXPORT_DIR)
stats = sec_tax["summary_stats"]

print("Security type taxonomy summary:")
for key, value in stats.items():
    print(f"  {key}: {value}")

display(sec_tax["summary"])
display(sec_tax["coverage"].head(25))
print("\nReview / unknown titles (sample):")
display(sec_tax["review"].head(20))

## Insider Role Taxonomy (Phase 2.4)

Classifies `RPTOWNER_TITLE` (and parses `RPTOWNER_RELATIONSHIP` flags) using `configs/normalization/officer_title_map.csv`.

Role weights are **informational priors** for future insider scoring — not predictive alpha. CEOs/CFOs often carry stronger information content, but that is tested in later phases.

In [ ]:
from src.utils.role_taxonomy import load_role_taxonomy

role_map = load_role_taxonomy()
display(role_map.head(20))

role_tax = audit.analyze_role_taxonomy(conn, export_dir=EXPORT_DIR)
stats = role_tax["summary_stats"]

print("Role taxonomy summary:")
for key, value in stats.items():
    print(f"  {key}: {value}")

display(role_tax["summary"].head(20))
display(role_tax["coverage"].head(20))
print("\nRelationship flag breakdown:")
display(role_tax["relationship_summary"].head(15))
print("\nReview / unknown roles (sample):")
display(role_tax["review"].head(15))

## Section 5 — Ownership analysis

Direct vs indirect ownership (`DIRECT_INDIRECT_OWNERSHIP`) and `NATURE_OF_OWNERSHIP` labels.

In [7]:
ownership = audit.analyze_ownership(conn)
audit.export_csv(ownership["ownership_distribution"], "ownership_distribution.csv", EXPORT_DIR)

display(ownership["direct_indirect"])
display(ownership["nature_of_ownership"].head(20))

12:49:50 | INFO | exported ownership_distribution.csv (52 rows)


,ownership_type,row_count,pct
0,D,8424113,84.8884
1,I,1499642,15.1116


,nature_of_ownership,row_count
0,See Footnote,113491
1,See footnote,97079
2,By Trust,91928
3,See Footnotes,55245
4,See footnotes,36780
5,By Spouse,26379
6,by Trust,25817
7,By Remainder of France Family Group,15098
8,By Limited Partnership II,13192
9,By Limited Partnership I,12914


## Section 6 — Role / title analysis

Raw `RPTOWNER_TITLE` values — noisy variants and normalization group candidates.

In [8]:
titles = audit.analyze_officer_titles(conn)
audit.export_csv(titles["officer_title_distribution"], "officer_title_distribution.csv", EXPORT_DIR)

display(titles["officer_title_distribution"].head(30))
display(titles["ceo_like_titles"].head(15))
display(titles["normalization_groups"].head(15))

12:50:11 | INFO | exported officer_title_distribution.csv (100 rows)


,officer_title,row_count,pct
0,Chief Financial Officer,102614,4.2575
1,Chief Executive Officer,87777,3.6419
2,President and CEO,78033,3.2376
3,Executive Vice President,72542,3.0098
4,Senior Vice President,54978,2.2811
5,President & CEO,53341,2.2132
6,Vice President,43723,1.8141
7,President,38095,1.5806
8,See Remarks,37996,1.5765
9,Chief Operating Officer,37697,1.5641


,officer_title,row_count,pct,title_upper
1,Chief Executive Officer,87777,3.6419,CHIEF EXECUTIVE OFFICER
2,President and CEO,78033,3.2376,PRESIDENT AND CEO
3,Executive Vice President,72542,3.0098,EXECUTIVE VICE PRESIDENT
4,Senior Vice President,54978,2.2811,SENIOR VICE PRESIDENT
5,President & CEO,53341,2.2132,PRESIDENT & CEO
6,Vice President,43723,1.8141,VICE PRESIDENT
7,President,38095,1.5806,PRESIDENT
10,CEO,30266,1.2558,CEO
11,Chairman and CEO,24806,1.0292,CHAIRMAN AND CEO
13,Chairman & CEO,17255,0.7159,CHAIRMAN & CEO


,title_normalized,row_count,example_raw_title
0,CHIEF FINANCIAL OFFICER,107738,CHIEF FINANCIAL OFFICER
1,CHIEF EXECUTIVE OFFICER,91982,CHIEF EXECUTIVE OFFICER
2,PRESIDENT AND CEO,81282,PRESIDENT AND CEO
3,EXECUTIVE VICE PRESIDENT,76741,EXECUTIVE VICE PRESIDENT
4,SENIOR VICE PRESIDENT,57940,SENIOR VICE PRESIDENT
5,PRESIDENT & CEO,55613,PRESIDENT & CEO
6,VICE PRESIDENT,46123,VICE PRESIDENT
7,SEE REMARKS,43224,SEE REMARKS
8,PRESIDENT,40324,PRESIDENT
9,CHIEF OPERATING OFFICER,39378,CHIEF OPERATING OFFICER


## Section 7 — Missing value analysis

Null rates and completeness scores for key fields used in later cleaning.

In [9]:
missing = audit.analyze_missing_values(conn)
audit.export_csv(missing, "missing_value_report.csv", EXPORT_DIR)
audit.plot_missingness_heatmap(missing, EXPORT_DIR)

display(missing.sort_values("null_pct", ascending=False))

12:51:37 | INFO | exported missing_value_report.csv (8 rows)
12:51:37 | INFO | saved chart missingness_heatmap.png


,table_name,column_name,field_label,total_rows,null_count,null_pct,completeness_score
6,raw_nonderiv_trans,TRANS_PRICEPERSHARE,transactionpricepershare,7215506,363066,5.0317,94.9683
1,raw_reporting_owner,RPTOWNERNAME,rptownername,4772838,1217,0.0255,99.9745
4,raw_nonderiv_trans,TRANS_CODE,transactioncode,7215506,6,0.0001,99.9999
0,raw_submission,ISSUERTRADINGSYMBOL,ticker,4402307,0,0.0000,100.0000
3,raw_nonderiv_trans,TRANS_DATE,transactiondate,7215506,0,0.0000,100.0000
2,raw_reporting_owner,RPTOWNERCIK,rptownercik,4772838,0,0.0000,100.0000
5,raw_nonderiv_trans,TRANS_SHARES,transactionshares,7215506,0,0.0000,100.0000
7,raw_nonderiv_trans,SECURITY_TITLE,securitytitle,7215506,0,0.0000,100.0000


## Section 8 — Distribution analysis

Histograms (linear + log) for shares, price, implied value. Uses a random SQL sample to limit memory.

In [10]:
dist = audit.analyze_distributions(conn, sample_size=100_000)
audit.export_csv(dist["distribution_summary"], "distribution_summary.csv", EXPORT_DIR)
audit.plot_distribution_histograms(dist["numeric_sample"], EXPORT_DIR)

display(dist["distribution_summary"])

12:52:23 | INFO | exported distribution_summary.csv (3 rows)
12:52:24 | INFO | saved chart dist_transaction_shares_linear.png
12:52:24 | INFO | saved chart dist_transaction_shares_log.png
12:52:24 | INFO | saved chart dist_transaction_price_per_share_linear.png
12:52:24 | INFO | saved chart dist_transaction_price_per_share_log.png
12:52:24 | INFO | saved chart dist_implied_transaction_value_linear.png
12:52:25 | INFO | saved chart dist_implied_transaction_value_log.png


,metric,count,mean,median,p01,p25,p75,p99,max,min
0,transaction_shares,100000,1.440465e+05,2000.00,3.0,400.0000,8976.50,9.974353e+05,2.275000e+09,0.01
1,transaction_price_per_share,100000,4.097732e+03,18.94,0.0,2.0300,46.52,6.443694e+02,2.500000e+08,0.00
2,implied_transaction_value,100000,3.182111e+10,17433.65,0.0,1000.0343,117087.75,7.863444e+06,2.662974e+15,0.00


## Section 9 — Filing lag analysis

`filing_lag_days = filing_date - transaction_date` using `raw_submission` + `raw_nonderiv_trans`.

In [11]:
# Phase 2.5 fix: SEC dates are DD-MON-YYYY (e.g. 01-APR-2008), parsed in Python.
lag = audit.compute_filing_lag(conn, export_dir=EXPORT_DIR, chunk_size=100_000)

display(lag["lag_stats"])
display(lag["lag_buckets"])
display(lag["yearly_lag"].tail(10))
display(lag["bad_rows"].head(20))

12:52:25 | INFO | computing filing lag (chunk_size=100000, sample_mode=False)
12:55:46 | INFO | exported filing_lag_analysis.csv (1 rows)
12:55:46 | INFO | exported filing_lag_bad_rows_sample.csv (1000 rows)
12:55:46 | INFO | saved chart filing_lag_distribution.png
12:55:46 | INFO | saved chart filing_lag_median_by_year.png


,row_count,valid_lag_count,invalid_date_count,mean_lag_days,median_lag_days,min_lag_days,max_lag_days,pct_same_day,pct_1_2_days,pct_over_2_days,pct_negative_lag
0,7215506,7215506,0,15.8109,2.0,-12054,10950,8.3529,54.0637,37.5602,0.0232


,lag_bucket,row_count
0,0-2 days,4503675
1,11-30 days,139084
2,3-10 days,2195633
3,31-90 days,145997
4,90+ days,229446
5,negative,1671


,filing_year,row_count,avg_lag_days,median_lag_days
11,2017,292115,14.4567,2.0
12,2018,305934,14.3943,2.0
13,2019,293866,13.4107,2.0
14,2020,309873,11.9346,2.0
15,2021,368826,12.1250,2.0
16,2022,292273,13.7389,2.0
17,2023,285611,13.4240,2.0
18,2024,300344,16.9345,2.0
19,2025,284747,10.6916,2.0
20,2026,103733,14.2976,2.0


,ACCESSION_NUMBER,NONDERIV_TRANS_SK,TRANS_DATE_raw,FILING_DATE_raw,DOCUMENT_TYPE,TRANS_CODE,filing_lag_days,issue_category
0,0001181431-06-021111,4122038,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
1,0001181431-06-021111,4122033,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
2,0001181431-06-021111,4122032,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
3,0001181431-06-021111,4122031,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
4,0001181431-06-021111,4122030,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
5,0001181431-06-021111,4122035,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
6,0001181431-06-021111,4122036,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
7,0001181431-06-021111,4122037,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
8,0001181431-06-021111,4122034,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days
9,0001181431-06-021111,4122039,01-AUG-2005,31-MAR-2006,4,S,242,lag_over_30_days


## Section 10 — Yearly market activity

Filings, insider buys (P), sells (S), and unique issuers/owners per year.

In [12]:
yearly = audit.analyze_yearly_activity(conn)
audit.export_csv(yearly, "yearly_activity_summary.csv", EXPORT_DIR)
audit.plot_yearly_activity_charts(yearly, EXPORT_DIR)

display(yearly)

12:56:26 | INFO | exported yearly_activity_summary.csv (21 rows)
12:56:26 | INFO | saved chart yearly_filings.png
12:56:26 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
12:56:26 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
12:56:26 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
12:56:26 | INFO | saved chart yearly_buys_sells_stacked.png


,year,filings,insider_buys,insider_sells,unique_insiders,unique_companies
0,2006,NaN,56395,339797,77260,8957
1,2007,NaN,76764,442415,77982,8973
2,2008,NaN,115271,254194,71857,8528
3,2009,NaN,54954,105364,64951,7724
4,2010,NaN,38556,119903,64976,7368
5,2011,NaN,45420,98454,63438,7157
6,2012,NaN,38649,103870,61585,6633
7,2013,NaN,29492,103025,61510,6536
8,2014,NaN,34172,95629,62627,6520
9,2015,NaN,41743,81115,61912,6597


## Section 11 — Data quality flags

Counts of negative shares, zero prices, impossible/future dates, duplicates, and extreme values.

In [13]:
quality = audit.analyze_data_quality_flags(conn)
audit.export_csv(quality, "data_quality_flags.csv", EXPORT_DIR)

display(quality)

12:59:49 | INFO | exported data_quality_flags.csv (7 rows)


,flag_name,table_name,issue_count,description
0,negative_shares,raw_nonderiv_trans,0,TRANS_SHARES < 0
1,zero_price,raw_nonderiv_trans,1371887,TRANS_PRICEPERSHARE = 0
2,future_transaction_date,raw_nonderiv_trans,0,TRANS_DATE > today
3,future_filing_date,raw_submission,0,FILING_DATE > today
4,filing_before_transaction,raw_nonderiv_trans+raw_submission,0,Filing date before transaction date
5,duplicate_row_hash,raw_nonderiv_trans,0,Duplicate row_hash values
6,extreme_implied_value,raw_nonderiv_trans,1847,Implied value > 10B


## Section 12 — Executive summary

Auto-generated markdown from audit artifacts. Also available as `executive_summary.md` in the export folder.

In [14]:
summary_md = audit.generate_executive_summary(
    overview, filing, trans, missing, quality, lag
)
(EXPORT_DIR / "executive_summary.md").write_text(summary_md, encoding="utf-8")
display(Markdown(summary_md))

# Phase 2 Raw Data Audit — Executive Summary

*Generated: 2026-05-18T10:59:49.666354+00:00*

## Key findings

- **Total raw rows** (8 core tables): 37,154,054
- **Database size (MB)**: 15154.67
- **Source year coverage**: 2006 – 2026

## Schema consistency (Phase 1.2)

- Review `data/exports/schema_audit/` for header drift across quarterly folders.
- Raw SQLite tables preserve SEC column names as TEXT without coercion.

## Dominant transaction codes

- Focus codes tracked: P, S, A, M, F, D, G, C, X
- Top focus codes in sample: S, A, M, P, F

## Data quality issues

- **zero_price** (raw_nonderiv_trans): 1,371,887 — TRANS_PRICEPERSHARE = 0
- **extreme_implied_value** (raw_nonderiv_trans): 1,847 — Implied value > 10B

## Missingness hotspots

- `transactionpricepershare` (raw_nonderiv_trans): 5.0317% null
- `rptownername` (raw_reporting_owner): 0.0255% null
- `transactioncode` (raw_nonderiv_trans): 0.0001% null
- `ticker` (raw_submission): 0.0% null
- `transactiondate` (raw_nonderiv_trans): 0.0% null

## Filing lag

- Valid lag rows: 7,215,506.0
- Mean lag (days): 15.8109
- Median lag (days): 2.0
- Same-day filings: 8.3529%
- Negative lag: 0.0232%

## Recommended cleaning rules (next phase — not applied here)

- Parse SEC text dates into ISO dates; drop rows with unparseable dates.
- Cast numeric fields only after validating non-numeric footnote placeholders.
- Standardize `SECURITY_TITLE` and `RPTOWNER_TITLE` via normalization lookup tables.
- Flag or exclude rows with negative shares, zero prices, or filing-before-transaction.

## Recommended filters for clean transaction layer

- Restrict to Form 4 open-market-style codes (`P`, `S`) only after explicit review.
- Require non-null `TRANS_DATE`, `TRANS_CODE`, `ISSUERTRADINGSYMBOL`, and owner CIK.
- Join `raw_nonderiv_trans` → `raw_submission` → `raw_reporting_owner` on `ACCESSION_NUMBER`.
- Deduplicate on `row_hash` or business keys (`ACCESSION_NUMBER`, `NONDERIV_TRANS_SK`).


## Run all sections at once (optional)

Equivalent to executing every section above in one call:

## Audit Fixes and Clean Layer Preparation (Phase 2.5)

Fixes filing lag parsing, audits numeric castability, and previews draft normalization maps for the upcoming clean layer.

**Still not applied:** no row deletion, no P/S filter, no `clean_insider_transactions` table yet.

In [15]:
from src.analytics.debug_raw_fields import run_debug_inspection

debug_out = EXPORT_DIR / "debug_raw_fields"
debug = run_debug_inspection(conn, debug_out)
display(debug["inspection"].head(20))

13:01:48 | INFO | wrote debug exports to C:\Users\artkh\Downloads\INSIDER ALPHA LAB\insider-research\data\exports\raw_data_audit\debug_raw_fields



=== Debug inspection: C:\Users\artkh\Downloads\INSIDER ALPHA LAB\insider-research\data\exports\raw_data_audit\debug_raw_fields ===

raw_submission columns (20):
ACCESSION_NUMBER, FILING_DATE, PERIOD_OF_REPORT, DATE_OF_ORIG_SUB, NO_SECURITIES_OWNED, NOT_SUBJECT_SEC16, FORM3_HOLDINGS_REPORTED, FORM4_TRANS_REPORTED, DOCUMENT_TYPE, ISSUERCIK, ISSUERNAME, ISSUERTRADINGSYMBOL, REMARKS, AFF10B5ONE, source_file, source_year, source_quarter, source_folder, ingested_at, row_hash

raw_nonderiv_trans columns (34):
ACCESSION_NUMBER, NONDERIV_TRANS_SK, SECURITY_TITLE, SECURITY_TITLE_FN, TRANS_DATE, TRANS_DATE_FN, DEEMED_EXECUTION_DATE, DEEMED_EXECUTION_DATE_FN, TRANS_FORM_TYPE, TRANS_CODE, EQUITY_SWAP_INVOLVED, EQUITY_SWAP_TRANS_CD_FN, TRANS_TIMELINESS, TRANS_TIMELINESS_FN, TRANS_SHARES, TRANS_SHARES_FN, TRANS_PRICEPERSHARE, TRANS_PRICEPERSHARE_FN, TRANS_ACQUIRED_DISP_CD, TRANS_ACQUIRED_DISP_CD_FN, SHRS_OWND_FOLWNG_TRANS, SHRS_OWND_FOLWNG_TRANS_FN, VALU_OWND_FOLWNG_TRANS, VALU_OWND_FOLWNG_TRANS_FN,

,table_name,column_name,field_type,non_null_count,sample_values,parsed_numeric_samples,parse_success_in_sample,parsed_iso_samples
0,raw_submission,ACCESSION_NUMBER,text,4402307,"[0000001750-06-000002, 0000001750-06-000004, 0...","[None, None, None, None, None, None, None, Non...",0,NaN
1,raw_submission,FILING_DATE,date,4402307,"[01-APR-2008, 01-APR-2008, 01-APR-2008, 01-APR...",NaN,20,"[2008-04-01, 2008-04-01, 2008-04-01, 2008-04-0..."
2,raw_submission,PERIOD_OF_REPORT,date,4402307,"[30-MAR-2006, 31-MAR-2006, 31-DEC-2004, 30-MAR...",NaN,20,"[2006-03-30, 2006-03-31, 2004-12-31, 2006-03-3..."
3,raw_submission,DOCUMENT_TYPE,text,4402307,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, ...",19,NaN
4,raw_submission,ISSUERTRADINGSYMBOL,text,4402307,"[WGR, APA, GBTVK, FLOW, TCC, MWFS, CLAY, sigm,...","[None, None, None, None, None, None, None, Non...",0,NaN
5,raw_submission,ISSUERCIK,text,4402307,"[0000856716, 0000006769, 0000839621, 000071300...","[856716.0, 6769.0, 839621.0, 713002.0, 1022438...",20,NaN
6,raw_submission,DATE_OF_ORIG_SUB,date,116746,"[11-AUG-2004, 14-MAR-2006, 18-MAR-1999, 20-JUN...",NaN,20,"[2004-08-11, 2006-03-14, 1999-03-18, 2005-06-2..."
7,raw_nonderiv_trans,ACCESSION_NUMBER,text,7215506,"[0000001750-06-000002, 0000001750-06-000002, 0...","[None, None, None, None, None, None, None, Non...",0,NaN
8,raw_nonderiv_trans,TRANS_DATE,date,7215506,"[29-MAR-2006, 29-MAR-2006, 29-MAR-2006, 29-MAR...",NaN,20,"[2006-03-29, 2006-03-29, 2006-03-29, 2006-03-2..."
9,raw_nonderiv_trans,TRANS_CODE,text,7215500,"[A, A, A, A, A, A, A, A, A, A, A, A, A, A, A, ...","[None, None, None, None, None, None, None, Non...",0,NaN


In [16]:
numeric = audit.audit_numeric_fields(conn, export_dir=EXPORT_DIR, chunk_size=100_000)
display(numeric["numeric_casting_audit"])

sec_map = pd.read_csv(PROJECT_ROOT / "configs/normalization/security_title_map.csv")
title_map = pd.read_csv(PROJECT_ROOT / "configs/normalization/officer_title_map.csv")
display(sec_map)
display(title_map)

13:01:48 | INFO | auditing numeric fields in chunks of 100000
13:02:50 | INFO | exported numeric_casting_audit.csv (4 rows)


,table_name,column_name,total_rows,parseable_count,unparseable_count,null_count,zero_count,negative_count,p01,median,p99,max
0,raw_nonderiv_trans,TRANS_SHARES,7215506,7215506,0,0,17159,0,2.3199,2037.00,1.500000e+06,9.219850e+10
1,raw_nonderiv_trans,TRANS_PRICEPERSHARE,7215506,6852440,0,363066,1371887,0,0.0000,18.76,6.477500e+02,4.235000e+07
2,raw_nonderiv_trans,SHRS_OWND_FOLWNG_TRANS,7215506,7213227,0,2279,227957,0,0.0000,59497.50,4.125687e+07,4.272126e+09
3,raw_nonderiv_trans,VALU_OWND_FOLWNG_TRANS,7215506,2279,0,7213227,185,0,0.0000,169605.84,3.326271e+08,1.300000e+10


,raw_pattern,normalized_security_type,include_in_clean_mvp,notes
0,COMMON STOCK,COMMON_EQUITY,true,Exact match on uppercased title
1,COMMON SHARES,COMMON_EQUITY,true,NaN
2,CLASS A COMMON STOCK,COMMON_EQUITY,true,NaN
3,CLASS B COMMON STOCK,COMMON_EQUITY,true,NaN
4,ORDINARY SHARES,COMMON_EQUITY,true,NaN
5,RESTRICTED STOCK UNIT,STOCK_COMPENSATION,false,Compensation; exclude from open-market MVP
6,RESTRICTED STOCK UNITS,STOCK_COMPENSATION,false,NaN
7,STOCK OPTION,OPTION_DERIVATIVE,false,Derivative; handle in deriv layer
8,PHANTOM STOCK,STOCK_COMPENSATION,false,NaN
9,DEFERRED STOCK UNIT,STOCK_COMPENSATION,false,NaN


,raw_pattern,normalized_role,role_weight,notes
0,CEO,CEO,1.00,Exact match on uppercased title
1,CHIEF EXECUTIVE OFFICER,CEO,1.00,NaN
2,PRESIDENT AND CEO,CEO,1.00,NaN
3,PRESIDENT & CEO,CEO,1.00,NaN
4,CHAIRMAN AND CEO,CEO,1.00,NaN
5,CFO,CFO,0.90,NaN
6,CHIEF FINANCIAL OFFICER,CFO,0.90,NaN
7,COO,COO,0.80,NaN
8,CHIEF OPERATING OFFICER,COO,0.80,NaN
9,PRESIDENT,PRESIDENT,0.75,NaN


### Updated recommended cleaning rules (draft)

1. Parse dates with `src.utils.sec_parsing.parse_sec_date` (supports `DD-MON-YYYY`, `YYYY-MM-DD`, `YYYYMMDD`).
2. Parse numbers with `parse_sec_number`; treat footnote text as null, not zero.
3. Join `raw_nonderiv_trans` → `raw_submission` → `raw_reporting_owner` on `ACCESSION_NUMBER`.
4. Apply `configs/normalization/security_title_map.csv` and `officer_title_map.csv` (draft lookups).
5. Flag negative shares, zero prices, filing-before-transaction, extreme implied values.
6. Filter Form 4 `P`/`S` only in the clean layer after review.
7. Deduplicate on `row_hash` or (`ACCESSION_NUMBER`, `NONDERIV_TRANS_SK`).

In [17]:
# Uncomment to regenerate all exports and charts in one pass:
# results = audit.run_full_audit(conn, EXPORT_DIR, DB_PATH)
# print("Audit complete. See", EXPORT_DIR)

In [18]:
conn.close()